# Module 8: Deploy to Akamai LKE

The agent only runs on your laptop. In this module you ship it to Akamai Cloud on Linode
Kubernetes Engine (LKE). The agent is a self-contained orchestrator: it calls its documentation,
account, and pricing specialists as tools inside one process. So you deploy it like any service.
Package it into one image, run it as a Deployment, put a NodeBalancer in front, and keep sessions
in Object Storage so any pod can serve any conversation. This is the end of the workshop: the
agent you built, running on the cloud it talks about.

## Learning objectives
- Serve the orchestrator over HTTP and package it into one container image
- Push the image to GHCR
- Deploy it to LKE as a Deployment behind a NodeBalancer
- Keep config and secrets out of the image with a Kubernetes Secret
- Prove it survives: durable Object Storage sessions across pods

## Prerequisites
- Finished Modules 1 through 7
- Docker, `kubectl`, and an LKE cluster with its kubeconfig (this module assumes `kubectl` points at it)
- A GitHub account for GHCR, with a token that has `write:packages` (push) and `read:packages` (pull)
- The Object Storage bucket and keys from Module 4
- About 45 minutes

> NOTE: this module creates BILLED Akamai resources (an LKE cluster and a NodeBalancer). The last
> section tears them down. Do not leave them running.

References: [Linode Kubernetes Engine](https://techdocs.akamai.com/cloud-computing/docs/linode-kubernetes-engine) · [NodeBalancer on LKE](https://techdocs.akamai.com/cloud-computing/docs/load-balancing-on-a-cluster) · [Object Storage](https://techdocs.akamai.com/cloud-computing/docs/object-storage) · [GitHub Container Registry](https://docs.github.com/packages)

## Deployment design basics

The orchestrator does its multi-agent work in process: it calls the specialists as Strands tools,
all inside one Python program. That means there is nothing exotic to deploy. It is one HTTP
service, and you run it on Kubernetes the way you run any service.

- A **Deployment** runs two replicas of the agent image, and restarts a pod if it dies.
- A **LoadBalancer Service** provisions an Akamai NodeBalancer and spreads traffic across the pods.
- A **Secret** holds the token, the model endpoint, and the Object Storage keys, so none of it is
  baked into the image.
- **Object Storage sessions** (Module 4) mean any pod can serve any conversation, so the
  NodeBalancer is free to send a follow-up request to a different pod.

![Build the image, push to GHCR, then a Deployment of two orchestrator pods behind a NodeBalancer, with sessions in Object Storage](../images/08_deploy_architecture.png)

## 1. Setup

Confirm your tools and that `kubectl` points at your LKE cluster, then make a namespace for the
agent.

In [ ]:
%%bash
docker version --format 'docker: {{.Server.Version}}' 2>/dev/null || echo "docker: start Docker Desktop"
kubectl config current-context
kubectl create namespace akamai-sa-agent --dry-run=client -o yaml | kubectl apply -f -

## 2. The agent as an HTTP service

`src/api.py` serves the agent over HTTP: `POST /invoke` runs a turn, `GET /healthz` is the probe,
`GET /tools` lists the tools. Set `AGENT_MODE=orchestrator` and it serves the Module 7 orchestrator
instead of the flat agent. It is stateless per request, with memory in the session backend, so it
scales. Run it locally and call it, the same way the cluster will.

Start it in a terminal from the repo root:

```bash
AGENT_MODE=orchestrator SESSION_BACKEND=memory \
  DOCS_INDEX_PATH=workshop/07_multi_agent/data/llms.txt \
  uvicorn api:app --app-dir src --host 0.0.0.0 --port 8080
```

Then call it:

In [ ]:
%%bash
curl -s localhost:8080/healthz
echo
curl -s -XPOST localhost:8080/invoke -H 'content-type: application/json' \
  -d '{"message":"What model and endpoint are you running on?"}' | python -m json.tool

## 3. Package it: the container image

`workshop/08_deploy_to_lke/Dockerfile` builds the image. It installs the sessions extra (boto3 for
Object Storage), bakes in `akamai-cloud-mcp` and the docs index, sets `AGENT_MODE=orchestrator`,
and serves `api:app` on port 8080. Build it and push it to GHCR.

In [ ]:
%%bash
# Set your GHCR image, then build and push. The token needs write:packages.
export IMAGE="ghcr.io/REPLACE_your_org/akamai-sa-agent:latest"
export GITHUB_USER="REPLACE_you"
export GITHUB_TOKEN="REPLACE_ghp_token"
# Builds from the repo root so the image can copy in src/ and the docs index.
./workshop/08_deploy_to_lke/scripts/build_and_push.sh

## 4. Secrets

The agent reads its config from a Secret: the Linode token, the vLLM endpoint, and the Object
Storage keys. The script builds it from your `.env` so the keys never land in a tracked file. Add
an image pull secret too, so the cluster can pull your private GHCR image.

In [ ]:
%%bash
# App secret, built from your .env so the keys never land in a tracked file.
./workshop/08_deploy_to_lke/scripts/make_secret.sh

# Image pull secret: the cluster pulls a private image, so it needs registry creds.
export GITHUB_USER="REPLACE_you"
export GITHUB_TOKEN="REPLACE_ghp_token"
./workshop/08_deploy_to_lke/scripts/ghcr_pull_secret.sh

## 5. Deploy

`manifests/deployment.yaml` runs two replicas and reads its config from the Secret.
`manifests/service.yaml` is a LoadBalancer Service, which on LKE provisions a NodeBalancer. Edit
the `image:` line in the Deployment to your image, then apply both and wait for the external IP.

> NOTE: the NodeBalancer takes a few minutes to come up. The external IP stays blank until then,
> and the agent is not reachable through it until the IP appears. Watch for it before you curl.

In [ ]:
%%bash
kubectl apply -f workshop/08_deploy_to_lke/manifests/deployment.yaml
kubectl apply -f workshop/08_deploy_to_lke/manifests/service.yaml

kubectl -n akamai-sa-agent rollout status deploy/akamai-sa-agent
kubectl -n akamai-sa-agent get pods
# Wait for the NodeBalancer to get an external IP:
kubectl -n akamai-sa-agent get svc akamai-sa-agent -w

Once the Service shows an external IP, call the agent through the NodeBalancer:

In [ ]:
%%bash
LB=$(kubectl -n akamai-sa-agent get svc akamai-sa-agent -o jsonpath='{.status.loadBalancer.ingress[0].ip}')
curl -s -XPOST "http://$LB/invoke" -H 'content-type: application/json' \
  -d '{"message":"what region are you running in?"}' | python -m json.tool

Now that it is on a node, `deployed_region` reads the Linode Metadata Service and answers a real
region, not the "running off-cluster" answer it gave on your laptop. The agent runs on the cloud
it talks about.

## 6. Prove it survives

This is where Object Storage sessions (Module 4) earn their place. If the agent kept memory in
process, deleting a pod would drop the conversation, and behind a NodeBalancer your next request
could land on a different pod that never saw it. Because the session lives in Object Storage, it
does not.

Show the gap close. Tell the agent something on a `session_id`, delete the pod that handled it,
then ask again. A fresh pod rehydrates the same session from Object Storage and the conversation
continues.

In [ ]:
%%bash
LB=$(kubectl -n akamai-sa-agent get svc akamai-sa-agent -o jsonpath='{.status.loadBalancer.ingress[0].ip}')
# Tell the agent a fact, on a named session.
curl -s -XPOST "http://$LB/invoke" \
  -d '{"message":"my cluster is named atlas, remember that","session_id":"demo"}' | python -m json.tool
# Delete the pod that handled it; the Deployment starts a fresh one.
kubectl -n akamai-sa-agent delete pod -l app=akamai-sa-agent --wait=false
# Ask again on the same session: a different pod rehydrates it from Object Storage.
curl -s -XPOST "http://$LB/invoke" \
  -d '{"message":"what is my cluster named?","session_id":"demo"}' | python -m json.tool

The second answer still knows the cluster is named `atlas`, even though the pod that first heard it
is gone. The conversation lives in Object Storage, not in the pod, so any replica picks it up.

## 7. Things to know

- **Writes stay gated.** The approval gate from Module 3 rides along: writes are denied by default,
  and an HTTP caller approves by re-sending the request with the `approve` token, never by letting
  the model self-grant. The deployed agent reads and proposes; a write needs that round-trip.
- **TLS and auth are the next step.** The Service is plain HTTP and the endpoint is open. Before you
  expose it for real, terminate TLS on the NodeBalancer (Linode Service annotations plus a cert) and
  put auth in front, since this agent reads your account. TLS encrypts the traffic; auth says who is
  allowed to call it.
- **Cost and teardown.** The LKE cluster and the NodeBalancer are billed while they exist. Tear
  down when you are done:

  ```bash
  kubectl delete -f workshop/08_deploy_to_lke/manifests/service.yaml      # removes the NodeBalancer
  kubectl delete -f workshop/08_deploy_to_lke/manifests/deployment.yaml
  # then delete the LKE cluster in the Cloud Manager, and the sessions bucket with
  # workshop/04_memory_and_sessions/scripts/delete_bucket.py
  ```
- **Scaling.** More replicas serve more requests, but they all call the one vLLM endpoint, which
  serves requests serially on its GPU. Scale the model, not just the agent.

## Try it yourself

1. Scale to three replicas with `kubectl -n akamai-sa-agent scale deploy/akamai-sa-agent --replicas=3`
   and confirm a conversation continues across pods.
2. Ask the deployed agent to diagram its own LKE cluster. It reads the cluster it is running on.
3. Point the agent at a second account by changing the Secret and restarting the rollout. Same
   image, different data.

## Summary

- The orchestrator does its multi-agent work in one process, so you deploy it like any HTTP
  service: a Deployment, a LoadBalancer Service for the NodeBalancer, and a Secret for config.
- Object Storage sessions make it horizontally scalable: any pod serves any conversation.
- The approval gate and the read-only MCP reads carry into production unchanged.
- That is the whole workshop, deployed. You built an agent from a raw HTTP call, gave it real data,
  guardrails, memory, diagrams, evals, and a team of specialists, and shipped it to Akamai Cloud.

## Done

You built and deployed a real agent on Akamai Cloud. Everything here maps to the code in this
repo's `src/`. The agent now runs as a service on the cloud it talks about.